# Introduction
This notebook is a clone of the run_quantize.py file for mimicking and debugging experiments without using seml and slurm

## 1. Initial Setup

In [1]:
print("Hello World")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

Hello World
Free GPU Memory (GB): 15.2344


In [2]:
# Setting up environment
print("\n################################")
print("Setting up environment...")
print("################################\n")

import os
#os.chdir('..')
print("Current Working Directory ", os.getcwd())
import sys
sys.path.append("../") # Add directory containing src/data to path

import importlib
import src  # Assuming src is the package name

# Reload the src module after making changes
importlib.reload(src)

%load_ext autoreload
%autoreload 2

import seml
import re
import shutil

os.environ["TOKENIZERS_PARALLELISM"] = "false"  # Disables parallelism to remove transformers warning

print("\n################################")
print("Setting up cache paths...")
print("################################\n")

os.environ["MKL_SERVICE_FORCE_INTEL"] = "1"
CACHE_PATH = "/nfs/students/daro/.cache/huggingface"
HUB_PATH = "/nfs/students/daro/.cache/huggingface/hub/"

if not os.path.exists(HUB_PATH):
    os.makedirs(HUB_PATH)
    print(f"Creating huggingface hub path at {HUB_PATH}")
    
print(f"Setting cache path to {CACHE_PATH}")
os.environ["TORCH_HOME"] = CACHE_PATH
os.environ["HF_HOME"] = CACHE_PATH

import torch
torch.hub.set_dir(CACHE_PATH)
with torch.no_grad():
    torch.cuda.empty_cache()
    
import logging
logger = logging.getLogger("quant_logger")
    
!cat /proc/meminfo | awk '/MemTotal/ {total=$2} /MemFree/ {free=$2} /MemAvailable/ {available=$2} END {printf "MemTotal: %.2f GB\nMemFree: %.2f GB\nMemAvailable: %.2f GB\n", total/1024/1024, free/1024/1024, available/1024/1024}'
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

print("\n################################")
print("Setting up cuda devices...")
print("################################\n")

if torch.cuda.is_available():
    print("CUDA device is available!")
    # Get the number of available CUDA devices
    num_cuda_devices = torch.cuda.device_count()
    print(f"Number of CUDA devices: {num_cuda_devices}")
    
    # Loop through available devices and get name
    for device_id in range(num_cuda_devices):
        device = torch.device(f"cuda:{device_id}")
        name = torch.cuda.get_device_name(device)
        print(f"  - CUDA Device {device_id+1}: {name}")
else:
    print("CUDA device is not available.")
    
print("\n################################")
print("Authentication with Hugging Face...")
print("################################\n")

import os
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv()
huggingface_token = os.getenv('HUGGINGFACE_TOKEN')

if huggingface_token is None:
    raise ValueError("Please set the HUGGINGFACE_TOKEN environment variable.")
else:
    print("Hugging Face token loaded successfully.")

login(token=huggingface_token, add_to_git_credential=True)
print("Successfully authenticated with the Hugging Face API.")

print("\n################################")
print("Setting up GPU memory usage list...")
print("################################\n")
# Global list to store GPU memory usage
from src.evaluations.evaluate_memory import record_gpu_memory
gpu_memory_usage = {}
record_gpu_memory(gpu_memory_usage=gpu_memory_usage, context="Warm up notebook")


################################
Setting up environment...
################################

Current Working Directory  /nfs/homedirs/daro/git/quantization-reliability
Initializing src package
Initializing src package

################################
Setting up cache paths...
################################

Setting cache path to /nfs/students/daro/.cache/huggingface
MemTotal: 1007.71 GB
MemFree: 30.04 GB
MemAvailable: 969.03 GB
Free GPU Memory (GB): 17.6406

################################
Setting up cuda devices...
################################

CUDA device is available!
Number of CUDA devices: 1
  - CUDA Device 1: NVIDIA A100-PCIE-40GB

################################
Authentication with Hugging Face...
################################

Hugging Face token loaded successfully.
Token is valid (permission: write).
Your token has been saved in your configured git credential helpers (store).
Your token has been saved to /nfs/students/daro/.cache/huggingface/token
Login successful

## 2. SEML Pipeline

## 2.1 Response Generator

In [3]:
exp_id = "10-30-1-test"

## 2.2 Pipeline

### Set up the experiment

In [4]:
import shutil
import re
import os
import subprocess
import tempfile
import logging

logger = logging.getLogger("quant_logger")

print("Setting up cache paths...")

# To avoid the following problem when running seml (see https://github.com/pytorch/pytorch/issues/37377)
os.environ["MKL_SERVICE_FORCE_INTEL"] = "1"
# Disables parallelism to remove transformers warning
os.environ["TOKENIZERS_PARALLELISM"] = "false"
CACHE_PATH = "/nfs/students/daro/.cache/huggingface"
HUB_PATH = os.path.join(CACHE_PATH, "hub")

if not os.path.exists(HUB_PATH):
    os.makedirs(HUB_PATH)
    print(f"Creating huggingface hub path at {HUB_PATH}")
    
print(f"Setting cache path to {CACHE_PATH}")
os.environ["TORCH_HOME"] = CACHE_PATH
os.environ["HF_HOME"] = CACHE_PATH
os.environ["HUGGINGFACE_HUB_CACHE"] = CACHE_PATH
os.environ["HUGGINGFACE_ASSETS_CACHE"] = CACHE_PATH
os.environ["TRANSFORMERS_CACHE"] = CACHE_PATH

import time
import random
import torch

torch.hub.set_dir(CACHE_PATH)

# Empty the cache
with torch.no_grad():
    torch.cuda.empty_cache()
    
print("Setting up working directory...")
print(f"Current Working Directory: {os.getcwd()}")
import sys
sys.path.append("../")

import os
import sys
import importlib.util   

import importlib
import src
importlib.reload(src)

logging.info("Setting up working directory...")

# HuggingFace authentication
from dotenv import load_dotenv
from huggingface_hub import login

logging.info("Authenticating Hugging Face...")

load_dotenv()
huggingface_token = os.getenv('HUGGINGFACE_TOKEN')
if huggingface_token is None:
    raise ValueError(
        f"Please set the HUGGINGFACE_TOKEN environment variable."
        f"Looking in {os.path.join(os.getcwd(), '.env')}"
    )
else:
    logging.info("Hugging Face token loaded successfully.")
login(token=huggingface_token, add_to_git_credential=True)

from src.evaluations.evaluate_reliability import evaluate_reliability

def validate_typo_config(typo_type, typo_intensity):
    """
    Validates the typo configuration based on the categories defined in the YAML.
    Returns True if the configuration is valid, False otherwise.
    """
    # Base case validation
    if typo_type == "none":
        return typo_intensity == 0
    
    # Specific perturbations validation
    if typo_type in ["word_remove_punctuation", "word_synonym"]:
        return typo_intensity == 1
    
    # Standard perturbations validation
    standard_perturbation_types = [
        "char_insertion", "char_deletion", "char_replacement",
        "char_repetition", "char_swapping", "word_CMW",
        "char_LCC", "char_insert_noise", "word_repeat",
        "char_substitution", "word_emoji", "word_internet_slang",
        "word_phrase_translation", "word_context_aware_insertion",
        "word_keyword_only"
    ]
    
    if typo_type in standard_perturbation_types:
        return typo_intensity in [1, 2, 3]
    
    return False

def run_evaluate(
    # Exp ID
    exp_id: str,
    save_excel: bool = True,
    num_excel_rows: int = 20,
    batch_size: int = 32,
    
    # Reliability dataset parameters
    seed=123,
    max_new_tokens=25,
    temperature=0.1,
    use_beam_search=False,
    strategy="Direct Completion",
    dataset_name="",
    typo_type="none",
    typo_intensity=0,
    n_repeats=10,
    n_beams=5,
    max_relations=None,
    max_entries=None,
    
    # Model parameters
    seed_model=123,
    model_name="",
    model_path="",
    device="cuda",
    cache_path=CACHE_PATH
):
    # Validate typo configuration
    if not validate_typo_config(typo_type, typo_intensity):
        raise ValueError(
            f"Invalid typo configuration: type={typo_type}, intensity={typo_intensity}. "
            f"Please check the configuration categories in the YAML file."
        )

    ##################
    ## Print config ##
    ##################
    print("Received the following configuration:")
    print(f"  Seed: {seed}")
    print(f"  Max new tokens: {max_new_tokens}")
    print(f"  Temperature: {temperature}")
    print(f"  Use beam search: {use_beam_search}")
    print(f"  Strategy: {strategy}")
    print(f"  Dataset name: {dataset_name}")
    print(f"  Typo type: {typo_type}")
    print(f"  Typo intensity: {typo_intensity}")
    print(f"  Number of repeats: {n_repeats}")
    print(f"  Number of beams: {n_beams}")
    print(f"  Max relations: {max_relations}")
    print(f"  Max entries: {max_entries}")
    print(f"  Seed model: {seed_model}")
    print(f"  Model name: {model_name}")
    print(f"  Model path: {model_path}")
    print(f"  Device: {device}")

    results = evaluate_reliability(
        exp_id=exp_id,
        model_name=model_name,
        dataset_name=dataset_name,
        typo_type=typo_type,
        typo_intensity=typo_intensity,
        strategy=strategy,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        use_beam_search=use_beam_search,
        n_repeats=n_repeats,
        n_beams=n_beams,
        max_relations=max_relations,
        max_entries=max_entries,
        save_excel=save_excel,
        num_excel_rows=num_excel_rows,
        cache_dir=cache_path,
        verbose=False,
        batch_size=batch_size
    )

    return results

# if __name__ == "__main__":
#     # Example usage
#     result = run_evaluate(
#         exp_id="test-run",
#         model_name="Llama-3-8B",
#         dataset_name="P17",
#         taxonomy_type="0",
#         strategy="Direct Completion",
#         max_entries=20,
#     )http://localhost:8008/tree?taoken=1f333809e711c109b6a50292336de304eefc47b99b6073c9
#     print(result)

Setting up cache paths...
Setting cache path to /nfs/students/daro/.cache/huggingface
Setting up working directory...
Current Working Directory: /nfs/homedirs/daro/git/quantization-reliability
Initializing src package
Token is valid (permission: write).
Your token has been saved in your configured git credential helpers (store).
Your token has been saved to /nfs/students/daro/.cache/huggingface/token
Login successful


/nfs/students/daro/miniconda3/envs/env-quant-rel-310/lib/python3.10/site-packages/transformers/utils/hub.py:124: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
2024-10-30 22:53:11.543269: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-10-30 22:53:11.556113: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-10-30 22:53:11.559744: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-10-30 22:53:11.569991: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimi

### Run configurations

In [5]:
import itertools
import logging
from typing import Dict, List, Any, Optional

logger = logging.getLogger("quant_logger")

# Fixed parameters from YAML
fixed_params = {
    'exp_id': "test-run-10-29",
    'save_excel': True,
    'num_excel_rows': 100,
    'device': 'cuda',
    'seed': 42,
    'n_repeats': 1,
    'n_beams': 5,
    'max_relations': None,
    'max_entries': None,
    'strategy': "Direct Completion",
    'use_beam_search': False,
    'max_new_tokens': 25,
    'temperature': 0.1
}

# Common grid parameters
common_grid_params = {
    'model_name': [
        'Llama-3-8B',
        'Llama-3-8B-QUANTO-local',
        'Llama-3-8B-QUANTO-CALIB-local',
        'Llama-3-8B-QUANTO-QAT-local'
    ],
    'batch_size': [32, 64, 128],
    'dataset_name': [
        'toy-qa-dataset',
        'coqa',
        'P101'
    ]
}

# Base case parameters
base_case_params = {
    'typo_type': ['none'],
    'typo_intensity': [0]
}

# Specific perturbations parameters
specific_perturbations_params = {
    'typo_type': ['word_remove_punctuation', 'word_synonym'],
    'typo_intensity': [1]
}

# Standard perturbations parameters
standard_perturbations_params = {
    'typo_type': ['word_emoji'],  # Add other types as needed
    'typo_intensity': [1, 2]
}

def generate_parameter_combinations(base_params: Dict[str, List[Any]], typo_params: Dict[str, List[Any]]) -> List[Dict[str, Any]]:
    """
    Generate all possible parameter combinations for a given set of base and typo parameters.
    
    Args:
        base_params: Dictionary containing base parameter options
        typo_params: Dictionary containing typo-specific parameter options
    
    Returns:
        List of dictionaries containing all possible parameter combinations
    """
    # Combine base and typo parameters
    all_params = {**base_params, **typo_params}
    
    # Get all parameter names and their possible values
    param_names = list(all_params.keys())
    param_values = [all_params[name] for name in param_names]
    
    # Generate all combinations
    combinations = list(itertools.product(*param_values))
    
    # Convert to list of dictionaries
    return [dict(zip(param_names, combo)) for combo in combinations]

def run_grid_search(max_combinations: Optional[int] = None) -> List[Dict[str, Any]]:
    """
    Run grid search evaluation with all parameter combinations.
    
    Args:
        max_combinations: Optional maximum number of combinations to evaluate
    
    Returns:
        List of dictionaries containing results and parameters for each combination
    """
    # Generate all parameter combinations
    all_combinations = []
    
    # Base case combinations
    base_combinations = generate_parameter_combinations(common_grid_params, base_case_params)
    all_combinations.extend(base_combinations)
    
    # Specific perturbations combinations
    specific_combinations = generate_parameter_combinations(common_grid_params, specific_perturbations_params)
    all_combinations.extend(specific_combinations)
    
    # Standard perturbations combinations
    standard_combinations = generate_parameter_combinations(common_grid_params, standard_perturbations_params)
    all_combinations.extend(standard_combinations)
    
    # Limit combinations if specified
    if max_combinations is not None:
        all_combinations = all_combinations[:max_combinations]
    
    results = []
    total_combinations = len(all_combinations)
    
    print(f"Starting grid search with {total_combinations} combinations")
    
    for i, params in enumerate(all_combinations):
        print(f"Running combination {i+1}/{total_combinations}")
        print("Parameters:")
        for k, v in params.items():
            print(f"  {k}: {v}")
        
        # Combine fixed parameters with current combination
        eval_params = {**fixed_params, **params}
        
        try:
            result = run_evaluate(**eval_params)
            
            # Store results with parameters
            results.append({
                'parameters': params,
                'result': result
            })
            
            print(f"Combination {i+1} completed successfully")
        except Exception as e:
            logger.error(f"Error in combination {i+1}: {str(e)}")
            results.append({
                'parameters': params,
                'result': None,
                'error': str(e)
            })
    
    return results

if __name__ == "__main__":
    # Configure logging
    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
    )
    
    # Run grid search with optional limit
    max_combinations = 2  # Adjust as needed
    results = run_grid_search(max_combinations)
    
    # Process and display results
    print("\nGrid Search Results:")
    for i, result in enumerate(results):
        print(f"\nResult {i+1}:")
        print("Parameters:")
        for k, v in result['parameters'].items():
            print(f"  {k}: {v}")
        
        if result['result'] is not None:
            print("Result metrics:")
            for k, v in result['result'].items():
                print(f"  {k}: {v}")
        else:
            print(f"Error: {result.get('error', 'Unknown error')}")

Starting grid search with 2 combinations
Running combination 1/2
Parameters:
  model_name: Llama-3-8B
  batch_size: 32
  dataset_name: toy-qa-dataset
  typo_type: none
  typo_intensity: 0
Received the following configuration:
  Seed: 42
  Max new tokens: 25
  Temperature: 0.1
  Use beam search: False
  Strategy: Direct Completion
  Dataset name: toy-qa-dataset
  Typo type: none
  Typo intensity: 0
  Number of repeats: 1
  Number of beams: 5
  Max relations: None
  Max entries: None
  Seed model: 123
  Model name: Llama-3-8B
  Model path: 
  Device: cuda


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
ERROR:quant_logger:Error in combination 1: too many values to unpack (expected 2)


Batch 1/1
Running combination 2/2
Parameters:
  model_name: Llama-3-8B
  batch_size: 32
  dataset_name: coqa
  typo_type: none
  typo_intensity: 0
Received the following configuration:
  Seed: 42
  Max new tokens: 25
  Temperature: 0.1
  Use beam search: False
  Strategy: Direct Completion
  Dataset name: coqa
  Typo type: none
  Typo intensity: 0
  Number of repeats: 1
  Number of beams: 5
  Max relations: None
  Max entries: None
  Seed model: 123
  Model name: Llama-3-8B
  Model path: 
  Device: cuda


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Batch 1/499
Batch 2/499
Batch 3/499
Batch 4/499
Batch 5/499
Batch 6/499
Batch 7/499
Batch 8/499
Batch 9/499
Batch 10/499
Batch 11/499
Batch 12/499
Batch 13/499
Batch 14/499
Batch 15/499
Batch 16/499
Batch 17/499
Batch 18/499
Batch 19/499
Batch 20/499
Batch 21/499
Batch 22/499
Batch 23/499
Batch 24/499
Batch 25/499
Batch 26/499
Batch 27/499
Batch 28/499
Batch 29/499
Batch 30/499
Batch 31/499
Batch 32/499
Batch 33/499
Batch 34/499
Batch 35/499
Batch 36/499
Batch 37/499
Batch 38/499
Batch 39/499
Batch 40/499
Batch 41/499
Batch 42/499
Batch 43/499
Batch 44/499
Batch 45/499
Batch 46/499
Batch 47/499
Batch 48/499
Batch 49/499
Batch 50/499
Batch 51/499
Batch 52/499
Batch 53/499
Batch 54/499
Batch 55/499
Batch 56/499
Batch 57/499
Batch 58/499
Batch 59/499
Batch 60/499
Batch 61/499
Batch 62/499
Batch 63/499
Batch 64/499
Batch 65/499
Batch 66/499
Batch 67/499
Batch 68/499
Batch 69/499
Batch 70/499
Batch 71/499
Batch 72/499
Batch 73/499
Batch 74/499
Batch 75/499
Batch 76/499
Batch 77/499
Batch 78

## 2.3 Benchmark Batch Size

In [3]:
import time
import torch
from torch.utils.data import DataLoader, Dataset
from src.evaluations.evaluate_reliability import evaluate_reliability
from src.data.FKTC_datasets import load_dataset_from_name

class QADataset(Dataset):
    def __init__(self, qa_pairs):
        self.qa_pairs = qa_pairs

    def __len__(self):
        return len(self.qa_pairs)

    def __getitem__(self, idx):
        return self.qa_pairs[idx]

def benchmark_batch_size(model_name, dataset_name, batch_sizes=[16, 32, 64, 128, 256]):
    # Fixed parameters
    fixed_params = {
        'exp_id': "generate-batch-size-benchmark",
        'save_excel': False,
        'num_excel_rows': 100,
        'seed': 42,
        'n_repeats': 5,
        'n_beams': 5,
        'max_entries': None,
        'max_new_tokens': 25,
        'temperature': 0.1,
        'use_beam_search': False,
        'strategy': "Direct Completion",
        'typo_type': "none",
        'typo_intensity': 0
    }

    # Load dataset
    qa_dataset = load_dataset_from_name(
        dataset_name,
        max_relations=1,
        max_entries=fixed_params['max_entries'],
        typo_type=fixed_params['typo_type'],
        typo_intensity=fixed_params['typo_intensity']
    )

    results = []
    for batch_size in batch_sizes:
        print(f"Benchmarking batch size: {batch_size}")
        
        # Create DataLoader
        dataloader = DataLoader(QADataset(qa_dataset), batch_size=batch_size, shuffle=False)
        
        start_time = time.time()
        memory_start = torch.cuda.memory_allocated()
        
        # Run evaluation
        evaluate_reliability(
            exp_id=fixed_params['exp_id'],
            model_name=model_name,
            dataset_name=dataset_name,
            typo_type=fixed_params['typo_type'],
            typo_intensity=fixed_params['typo_intensity'],
            strategy=fixed_params['strategy'],
            max_new_tokens=fixed_params['max_new_tokens'],
            temperature=fixed_params['temperature'],
            use_beam_search=fixed_params['use_beam_search'],
            n_repeats=fixed_params['n_repeats'],
            n_beams=fixed_params['n_beams'],
            max_entries=fixed_params['max_entries'],
            save_excel=fixed_params['save_excel'],
            num_excel_rows=fixed_params['num_excel_rows'],
            batch_size=batch_size
        )
        
        end_time = time.time()
        memory_end = torch.cuda.memory_allocated()
        
        # Calculate total tokens processed
        total_tokens = sum(len(query.split()) + fixed_params['max_new_tokens'] for query, _ in qa_dataset) * fixed_params['n_repeats']
        
        results.append({
            'batch_size': batch_size,
            'runtime': end_time - start_time,
            'memory_used': (memory_end - memory_start) / 1024 / 1024,  # Convert to MB
            'tokens_per_second': total_tokens / (end_time - start_time)
        })
        
        print(f"Batch size {batch_size} completed")
        print(f"Runtime: {results[-1]['runtime']:.2f} seconds")
        print(f"Memory used: {results[-1]['memory_used']:.2f} MB")
        print(f"Tokens per second: {results[-1]['tokens_per_second']:.2f}")
        print("---")

    return results

# Example usage
model_name = "Llama-3-8B"
dataset_name = "P17"
benchmark_results = benchmark_batch_size(model_name, dataset_name)

# Print summary of results
print("\nBenchmark Results Summary:")
for result in benchmark_results:
    print(f"Batch Size: {result['batch_size']}")
    print(f"Runtime: {result['runtime']:.2f} seconds")
    print(f"Memory Used: {result['memory_used']:.2f} MB")
    print(f"Tokens per Second: {result['tokens_per_second']:.2f}")
    print("---")

# Determine optimal batch size (based on tokens per second)
optimal_batch_size = max(benchmark_results, key=lambda x: x['tokens_per_second'])['batch_size']
print(f"\nOptimal Batch Size: {optimal_batch_size}")

2024-10-14 21:29:55.168238: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-10-14 21:29:55.187527: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-10-14 21:29:55.193179: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-10-14 21:29:55.207291: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-10-14 21:29:56.610354: W tensorflow/compiler/tf2

Benchmarking batch size: 16


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Tokenizer model max length reduced from 1000000000000000019884624838656 to 2048 to fit in memory


Batch 1/59
Batch 2/59
Batch 3/59
Batch 4/59
Batch 5/59
Batch 6/59
Batch 7/59
Batch 8/59
Batch 9/59
Batch 10/59
Batch 11/59
Batch 12/59
Batch 13/59
Batch 14/59
Batch 15/59
Batch 16/59
Batch 17/59
Batch 18/59
Batch 19/59
Batch 20/59
Batch 21/59
Batch 22/59
Batch 23/59
Batch 24/59
Batch 25/59
Batch 26/59
Batch 27/59
Batch 28/59
Batch 29/59
Batch 30/59
Batch 31/59
Batch 32/59
Batch 33/59
Batch 34/59
Batch 35/59
Batch 36/59
Batch 37/59
Batch 38/59
Batch 39/59
Batch 40/59
Batch 41/59
Batch 42/59
Batch 43/59
Batch 44/59
Batch 45/59
Batch 46/59
Batch 47/59
Batch 48/59
Batch 49/59
Batch 50/59
Batch 51/59
Batch 52/59
Batch 53/59
Batch 54/59
Batch 55/59
Batch 56/59
Batch 57/59
Batch 58/59
Batch 59/59
Batch size 16 completed
Runtime: 296.46 seconds
Memory used: 8.12 MB
Tokens per second: 503.01
---
Benchmarking batch size: 32


...

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Tokenizer model max length reduced from 1000000000000000019884624838656 to 2048 to fit in memory


Batch 1/8
Batch 2/8
Batch 3/8
Batch 4/8
Batch 5/8
Batch 6/8
Batch 7/8
Batch 8/8
Batch size 128 completed
Runtime: 96.80 seconds
Memory used: 0.00 MB
Tokens per second: 1540.53
---
Benchmarking batch size: 256


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Tokenizer model max length reduced from 1000000000000000019884624838656 to 2048 to fit in memory


Batch 1/4
Batch 2/4
Batch 3/4
Batch 4/4
Batch size 256 completed
Runtime: 104.35 seconds
Memory used: 0.00 MB
Tokens per second: 1429.11
---

Benchmark Results Summary:
Batch Size: 16
Runtime: 296.46 seconds
Memory Used: 8.12 MB
Tokens per Second: 503.01
---
Batch Size: 32
Runtime: 172.66 seconds
Memory Used: 0.00 MB
Tokens per Second: 863.69
---
Batch Size: 64
Runtime: 114.84 seconds
Memory Used: 0.00 MB
Tokens per Second: 1298.55
---
Batch Size: 128
Runtime: 96.80 seconds
Memory Used: 0.00 MB
Tokens per Second: 1540.53
---
Batch Size: 256
Runtime: 104.35 seconds
Memory Used: 0.00 MB
Tokens per Second: 1429.11
---

Optimal Batch Size: 128


In [5]:
%history

print("Hello World")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'
# Setting up environment
print("\n################################")
print("Setting up environment...")
print("################################\n")

import os
#os.chdir('..')
print("Current Working Directory ", os.getcwd())
import sys
sys.path.append("../") # Add directory containing src/data to path

import importlib
import src  # Assuming src is the package name

# Reload the src module after making changes
importlib.reload(src)

%load_ext autoreload
%autoreload 2

import seml
import re
import shutil

os.environ["TOKENIZERS_PARALLELISM"] = "false"  # Disables parallelism to remove transformers warning

print("\n################################")
print("Setting up cache paths...")
print("################################\n")

os.environ["MKL_SERVICE_FORCE_INTEL"] = "1"
CACHE_PATH = "/nfs/students/daro/.cache/huggingface"
HUB_PATH = "/nfs/students/

In [4]:
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

Free GPU Memory (GB): 5.4375


### Count dataset rows

In [3]:
import os
import json

DATA_DIR = "/nfs/students/daro/data/MONITOR/FKTC"
DATA_FILES = [
    "P101", "P103", "P108", "P127", "P1376", "P1412", "P159", "P17", "P176", "P178",
    "P19", "P20", "P264", "P27", "P276", "P30", "P364", "P37", "P495", "P740"
]

def count_entries(file_path):
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            lines = f.readlines()
        # Subtract 1 to exclude the first line containing relations
        return len(lines) - 1
    except FileNotFoundError:
        print(f"File not found: {file_path}")
        return 0
    except json.JSONDecodeError:
        print(f"Error decoding JSON in file: {file_path}")
        return 0

def main():
    for dataset in DATA_FILES:
        file_path = os.path.join(DATA_DIR, f"{dataset}-subclass.json")
        entry_count = count_entries(file_path)
        print(f"Dataset {dataset}: {entry_count} entries")

if __name__ == "__main__":
    main()

Dataset P101: 696 entries
Dataset P103: 977 entries
Dataset P108: 383 entries
Dataset P127: 575 entries
Dataset P1376: 234 entries
Dataset P1412: 969 entries
Dataset P159: 967 entries
Dataset P17: 930 entries
Dataset P176: 982 entries
Dataset P178: 592 entries
Dataset P19: 944 entries
Dataset P20: 953 entries
Dataset P264: 429 entries
Dataset P27: 966 entries
Dataset P276: 959 entries
Dataset P30: 975 entries
Dataset P364: 856 entries
Dataset P37: 966 entries
Dataset P495: 909 entries
Dataset P740: 936 entries


## 3. Plot Results

In [ ]:
import os
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from fpdf import FPDF

def create_and_save_figures(df, plots_dir, model_name, strategy, max_new_tokens, temperature):
    plots = []
    metrics = ['Accuracy', 'AUCPR_sem']

    # Function to create a box plot
    def create_box_plot(data, x_column, y_column, title):
        fig = go.Figure()
        for x_value in sorted(data[x_column].unique()):
            subset = data[data[x_column] == x_value]
            fig.add_trace(go.Box(
                x=[str(x_value)] * len(subset),
                y=subset[y_column],
                name=str(x_value),
                boxpoints='all'
            ))
        fig.update_layout(
            title=title,
            xaxis_title=x_column,
            yaxis_title=y_column,
            height=600,
            width=1000
        )
        return fig

    # Taxonomy effects
    taxonomy_df = df[df['Typo Type'] == 'none']
    for metric in metrics:
        # Box plot for taxonomy effects
        fig = create_box_plot(
            taxonomy_df, 
            'Taxonomy Type', 
            metric, 
            f'{metric} by Taxonomy Type (Model: {model_name}, Strategy: {strategy})'
        )
        plot_file = os.path.join(plots_dir, f"plot_taxonomy_{metric.lower()}.png")
        fig.write_image(plot_file)
        plots.append((plot_file, f"{metric} by Taxonomy Type"))

        # Bar plot for mean taxonomy effects
        mean_values = taxonomy_df.groupby('Taxonomy Type')[metric].mean().reset_index()
        fig = go.Figure(data=[
            go.Bar(x=mean_values['Taxonomy Type'], y=mean_values[metric])
        ])
        fig.update_layout(
            title=f'Mean {metric} by Taxonomy Type (Model: {model_name}, Strategy: {strategy})',
            xaxis_title='Taxonomy Type',
            yaxis_title=f'Mean {metric}',
            height=600,
            width=1000
        )
        plot_file = os.path.join(plots_dir, f"plot_mean_taxonomy_{metric.lower()}.png")
        fig.write_image(plot_file)
        plots.append((plot_file, f'Mean {metric} by Taxonomy Type'))

    # Typo effects
    typo_df = df[df['Taxonomy Type'] == '0']
    for metric in metrics:
        # Box plot for typo effects
        fig = make_subplots(rows=1, cols=2, subplot_titles=('Typo Type', 'Typo Intensity'))
        
        fig.add_trace(
            create_box_plot(
                typo_df, 
                'Typo Type', 
                metric, 
                ''
            ).data[0],
            row=1, col=1
        )
        
        fig.add_trace(
            create_box_plot(
                typo_df, 
                'Typo Intensity', 
                metric, 
                ''
            ).data[0],
            row=1, col=2
        )
        
        fig.update_layout(
            title=f'{metric} by Typo Type and Intensity (Model: {model_name}, Strategy: {strategy})',
            height=600,
            width=1000
        )
        plot_file = os.path.join(plots_dir, f"plot_typo_{metric.lower()}.png")
        fig.write_image(plot_file)
        plots.append((plot_file, f"{metric} by Typo Type and Intensity"))

        # Heatmap for typo effects
        pivot_df = typo_df.pivot_table(values=metric, index='Typo Type', columns='Typo Intensity', aggfunc='mean')
        fig = go.Figure(data=go.Heatmap(
            z=pivot_df.values,
            x=pivot_df.columns,
            y=pivot_df.index,
            colorscale='Viridis'
        ))
        fig.update_layout(
            title=f'Mean {metric} by Typo Type and Intensity (Model: {model_name}, Strategy: {strategy})',
            xaxis_title='Typo Intensity',
            yaxis_title='Typo Type',
            height=600,
            width=1000
        )
        plot_file = os.path.join(plots_dir, f"plot_heatmap_typo_{metric.lower()}.png")
        fig.write_image(plot_file)
        plots.append((plot_file, f'Heatmap of Mean {metric} by Typo Type and Intensity'))

    return plots

def generate_pdf_report(plots, pdf_path, model_name, strategy, max_new_tokens, temperature):
    pdf = FPDF()
    pdf.set_auto_page_break(auto=True, margin=15)
    
    # Add title page
    pdf.add_page()
    pdf.set_font("Arial", 'B', size=16)
    pdf.cell(0, 10, "Taxonomy and Typo Effect Analysis", ln=True, align='C')
    pdf.set_font("Arial", size=12)
    pdf.cell(0, 10, f"Model: {model_name}", ln=True, align='C')
    pdf.cell(0, 10, f"Strategy: {strategy}", ln=True, align='C')
    pdf.cell(0, 10, f"Max New Tokens: {max_new_tokens}", ln=True, align='C')
    pdf.cell(0, 10, f"Temperature: {temperature}", ln=True, align='C')

    # Add plots and descriptions to the PDF
    for plot_file, desc in plots:
        pdf.add_page()
        pdf.set_font("Arial", 'B', size=14)
        pdf.multi_cell(0, 10, desc)
        pdf.ln(5)
        pdf.image(plot_file, w=pdf.w - 20)
        
        # Add some interpretation text (you may want to customize this based on the actual results)
        pdf.ln(10)
        pdf.set_font("Arial", size=10)
        if "Taxonomy" in desc:
            pdf.multi_cell(0, 5, "This plot shows how different taxonomy types affect the model's performance. " 
                                 "Higher values indicate better performance. Variations across taxonomy types " 
                                 "may suggest areas where the model is more or less reliable.")
        elif "Typo" in desc:
            pdf.multi_cell(0, 5, "This plot illustrates the impact of different typo types and intensities on " 
                                 "the model's performance. Lower scores for certain typo types or higher intensities " 
                                 "indicate areas where the model's reliability decreases.")

    pdf.output(pdf_path, "F")

def main(exp_id, model_name, strategy, max_new_tokens, temperature):
    # Load the Excel file
    file_path = f"results/reliability_eval/unified_scores_table_{exp_id}.xlsx"
    df = pd.read_excel(file_path)

    # Filter the dataframe to include only the specified model and parameters
    df = df[(df['Model Name'] == model_name) & 
            (df['Strategy'] == strategy) & 
            (df['Max New Tokens'] == max_new_tokens) & 
            (df['Temperature'] == temperature)]

    # Create a directory for saving plots
    plots_dir = f"plots/taxonomy_typo_eval_{exp_id}"
    os.makedirs(plots_dir, exist_ok=True)

    # Generate and save the figures
    plots = create_and_save_figures(df, plots_dir, model_name, strategy, max_new_tokens, temperature)

    # Generate the PDF report
    pdf_path = os.path.join(plots_dir, f"taxonomy_typo_evaluation_plots_{exp_id}.pdf")
    generate_pdf_report(plots, pdf_path, model_name, strategy, max_new_tokens, temperature)

    print(f"PDF report generated and saved as '{pdf_path}'")

if __name__ == "__main__":
    # Example usage
    main(
        exp_id="your-experiment-id",
        model_name="Your-Model-Name",
        strategy="Your-Strategy",
        max_new_tokens=25,
        temperature=0.1
    )